# KnotTransformer — Phase 2 Training (v2, hardened)

**Changes from v1:**
1. **GPU-aware evaluation** — fixes the silent failure where `evaluate_holdout`
   used CPU tensors against a GPU model, causing all in-training held-out
   metrics to be NaN. (This is why your previous run showed `inf` as best
   held-out ratio at the end.)
2. **Drive mounting is required**, not optional. Colab disconnects after 90
   min idle and wipes `/content`. Saving to Drive is the only way checkpoints
   survive long training runs.
3. **Lightweight per-epoch persistence + every-25-epoch full snapshot.**
   Per-epoch we save the small trajectory JSON; every 25 epochs we also copy
   the best checkpoints into a numbered snapshot so even mid-run you have a
   recoverable model.
4. **The trajectory plot is in its own cell that loads from disk**, so even
   if training is interrupted you can plot what you have so far.

If you're running this after losing a previous run, **don't worry about the
re-train cost** — on GPU the full 300 epochs is ~5–10 hours and you can let
it run unattended once Drive is mounted.

## 1 · GPU check

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
    print("VRAM total:", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Runtime → Change runtime type → T4 GPU")

## 2 · Upload the code bundle

Upload `colab_bundle.zip` (same one as before).

In [ ]:
from google.colab import files
import zipfile, os

print("Click 'Choose Files' and select colab_bundle.zip ...")
uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name, 'r') as z:
            z.extractall('/content')
        print(f"Extracted {name}")

for f in ['knot_transformer.py', 'curvature_layer_nd.py',
          'curvatureLayer.py', 'generalization_experiment.py',
          'phase1_generalization.pt', 'dataset_1k.pt']:
    p = f'/content/{f}'
    print(f"  {'✓' if os.path.exists(p) else '✗'}  {f}")

## 3 · Mount Google Drive (REQUIRED)

This is not optional this time. Without Drive your training is one disconnect
away from total loss.

If you get a `MessageError: credential propagation was unsuccessful`:
- Try an incognito/private window signed into Google
- Or enable third-party cookies for `accounts.google.com` and `colab.research.google.com`
- Then re-run this cell

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

CKPT_DIR = '/content/drive/MyDrive/knot_transformer_ckpts'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(f'{CKPT_DIR}/snapshots', exist_ok=True)
print(f"Saving checkpoints to: {CKPT_DIR}")
print(f"Snapshots subdir     : {CKPT_DIR}/snapshots")

## 4 · Training config

In [ ]:
# ─── Training hyperparameters ─────────────────────────────────────────
TARGET_EPOCHS  = 300
BATCH_SIZE     = 128
LR             = 1e-5
ENERGY_CAP     = 1000.0
EPS_BOUNDARY   = 0.02
GRAD_CLIP      = 1.0
HOLDOUT_EVERY  = 5      # evaluate on held-out N=6 every N epochs
HOLDOUT_N      = 200    # held-out set size during training
SNAPSHOT_EVERY = 25     # copy best checkpoints to snapshots/ every N epochs
SEED           = 20260525

DATA_PT          = '/content/dataset_1k.pt'
PHASE1_PT        = '/content/phase1_generalization.pt'
STATE_PT         = f'{CKPT_DIR}/phase2_state.pt'
BEST_HOLDOUT_PT  = f'{CKPT_DIR}/phase2_best_holdout.pt'
BEST_TRAIN_PT    = f'{CKPT_DIR}/phase2_best_trainloss.pt'
LATEST_PT        = f'{CKPT_DIR}/phase2_latest.pt'   # the most recent epoch
TRAJ_PT          = f'{CKPT_DIR}/phase2_trajectory.json'

print(f"Target epochs : {TARGET_EPOCHS}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Learning rate : {LR}")
print(f"Checkpoints   : {CKPT_DIR}")

## 5 · Imports + GPU device

In [ ]:
import sys, time, json
sys.path.insert(0, '/content')

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import knot_transformer as kt_mod
import curvature_layer_nd as cl
import curvatureLayer  # backward-compat shim
import generalization_experiment as ge

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Patch the modules to use the GPU
kt_mod.device = DEVICE
ge.device     = DEVICE

from knot_transformer import (
    KnotTransformer, BSplineDataset, collate_variable_length,
)
from generalization_experiment import gen_random_points

np.random.seed(SEED); torch.manual_seed(SEED)

## 6 · Load data and model

In [ ]:
payload = torch.load(DATA_PT, weights_only=False)
ds = BSplineDataset(payload['pts'], payload['heur'], task='curvature')
print(f"Training samples: {len(ds)}")
ns = [len(h) + 2 for h in payload['heur']]
for n_pts in sorted(set(ns)):
    print(f"  N={n_pts} ({n_pts-2} dof): {ns.count(n_pts)} samples")

# Held-out N=6 set
rng = np.random.default_rng(SEED)
_ = gen_random_points(2000, 4, rng)
_ = gen_random_points(2000, 5, rng)
_ = gen_random_points(500, 4, rng)
_ = gen_random_points(500, 5, rng)
holdout = gen_random_points(500, 6, rng)[:HOLDOUT_N]
print(f"\nHeld-out: {len(holdout)} N=6 samples (4 dof, UNSEEN during training)")

# Model + optimizer
model = KnotTransformer(
    d_model=64, nhead=4,
    num_encoder_layers=3, num_decoder_layers=3,
    dim_feedforward=256, dropout=0.1,
    max_points=10, max_knots=6,
).to(DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=LR)
print(f"\nModel params: {sum(p.numel() for p in model.parameters()):,}")

## 7 · Training loop + (fixed) held-out evaluation

**The fix:** `evaluate_holdout` now puts the input tensor on the model's
device. The v1 notebook ran the model on GPU but fed it a CPU tensor, which
threw silently inside `evaluate_on_test_set`'s `except Exception: pass`
block. The result: every held-out evaluation returned NaN, and the
"best held-out ratio" stayed at `inf`.

In [ ]:
def rescale_interior(knots, eps=EPS_BOUNDARY):
    return eps + (1.0 - 2.0 * eps) * knots


def train_one_epoch(model, loader, optim):
    model.train()
    epoch_capped = 0.0
    epoch_valid  = 0
    nan_skips    = 0
    raw_energies = []

    for batch in loader:
        points_2d  = batch['points'].to(DEVICE)
        point_mask = batch['point_mask'].to(DEVICE)
        num_knots  = batch['num_knots'].to(DEVICE)

        optim.zero_grad()
        predicted = model(points_2d, point_mask, num_knots)
        B = points_2d.shape[0]

        n_pts_per = (point_mask == False).sum(dim=1)
        groups = {}
        for b in range(B):
            n = int(n_pts_per[b].item())
            groups.setdefault(n, []).append(b)

        batch_energy_sum = []
        batch_n_valid = 0
        for n_pts, idxs in groups.items():
            K = n_pts - 2
            pts_g  = points_2d[idxs, :n_pts]
            pred_g = predicted[idxs, :K]
            interior_g = rescale_interior(pred_g)

            G = len(idxs)
            zeros = torch.zeros(G, 4, dtype=torch.double, device=DEVICE)
            ones  = torch.ones( G, 4, dtype=torch.double, device=DEVICE)
            full_kv = torch.cat([zeros, interior_g, ones], dim=1)
            pts_cl = pts_g.unsqueeze(-1)

            try:
                C   = cl.solve(pts_cl, full_kv)
                eng = cl.global_curvature(C, full_kv, 3, 100)
            except Exception:
                nan_skips += G
                continue

            finite = torch.isfinite(eng)
            nb = int((~finite).sum().item())
            if nb: nan_skips += nb
            n_ok = int(finite.sum().item())
            if n_ok == 0: continue

            eng_ok = eng[finite]
            eng_cap = torch.clamp(eng_ok, max=ENERGY_CAP)
            batch_energy_sum.append(eng_cap.sum())
            batch_n_valid += n_ok
            raw_energies.extend(eng_ok.detach().cpu().tolist())

        if not batch_energy_sum: continue
        loss = torch.stack(batch_energy_sum).sum() / batch_n_valid
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optim.step()
        epoch_capped += float(loss.detach()) * batch_n_valid
        epoch_valid  += batch_n_valid

    capped_mean = epoch_capped / max(epoch_valid, 1)
    raw_med = float(np.median(raw_energies)) if raw_energies else float('nan')
    raw_p99 = float(np.percentile(raw_energies, 99)) if raw_energies else float('nan')
    return capped_mean, raw_med, raw_p99, epoch_valid, nan_skips


# ─── THE FIX: GPU-aware held-out evaluation ──────────────────────────────
def evaluate_holdout_gpu(model, holdout_pts, n_dof=4):
    """Returns (win_rate, median_ratio) on held-out set.

    Unlike ge.evaluate_on_test_set, this function explicitly moves tensors
    to the model's device before predict(). That was the silent bug in v1.
    """
    model.eval()
    kappa_model, kappa_chordal = [], []
    better = valid = 0

    for points_np in holdout_pts:
        chord_knots = ge.chordal_interior_knots(points_np)

        pts_t = torch.tensor(points_np, dtype=torch.double).to(DEVICE)  # ← THE FIX
        with torch.no_grad():
            pred_knots = model.predict(pts_t, num_knots=n_dof)
        pred_knots_np = pred_knots.cpu().numpy()
        pred_eval = ge.rescale_knots_to_interior(pred_knots_np)

        k_m = ge.compute_curvature_batch(points_np, pred_eval)
        k_c = ge.compute_curvature_batch(points_np, chord_knots)
        if np.isnan(k_m) or np.isnan(k_c): continue

        kappa_model.append(k_m)
        kappa_chordal.append(k_c)
        if k_m <= k_c: better += 1
        valid += 1

    if not valid:
        return float('nan'), float('nan')
    win = better / valid
    ratio = float(np.median(kappa_model)) / float(np.median(kappa_chordal))
    return win, ratio


# Smoke-test: prove the eval works on the freshly built (pre-Phase-1) model
test_win, test_ratio = evaluate_holdout_gpu(model, holdout[:20])
print(f"Smoke test on 20 samples (untrained model): "
      f"win={test_win:.3f}, ratio={test_ratio:.4f}")
print("If you see real numbers (not nan), the eval is wired correctly.")

## 8 · Train

This cell saves to Drive after every epoch and snapshots checkpoints every 25
epochs. If the runtime disconnects, just **mount Drive again** (Cell 3) and
re-run this cell — it resumes from the last saved state.

In [ ]:
# Resume / fresh start
completed = 0
trajectory = []
best_holdout_ratio = float('inf')
best_train_loss    = float('inf')

if os.path.exists(STATE_PT):
    state = torch.load(STATE_PT, weights_only=False, map_location=DEVICE)
    model.load_state_dict(state['model'])
    optim.load_state_dict(state['opt'])
    completed = state['completed']
    trajectory = state['trajectory']
    best_holdout_ratio = state.get('best_holdout_ratio', float('inf'))
    best_train_loss    = state.get('best_train_loss',    float('inf'))
    print(f"Resumed at epoch {completed}/{TARGET_EPOCHS}")
    print(f"  Best held-out ratio so far: {best_holdout_ratio:.4f}")
    print(f"  Best train loss so far:     {best_train_loss:.4f}")
else:
    print("Fresh start — loading Phase 1 checkpoint")
    model.load_state_dict(torch.load(PHASE1_PT, map_location=DEVICE))

loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                    collate_fn=collate_variable_length)

print(f"\nTraining epochs {completed+1}..{TARGET_EPOCHS}")
print(f"  Batch {BATCH_SIZE}, lr {LR}, energy cap {ENERGY_CAP}")
print(f"  Held-out eval every {HOLDOUT_EVERY} epochs on {HOLDOUT_N}-sample N=6 set")
print(f"  Snapshot every {SNAPSHOT_EVERY} epochs to {CKPT_DIR}/snapshots/")
print('─' * 78)

t0 = time.time()
for ep in range(completed + 1, TARGET_EPOCHS + 1):
    capped, raw_med, raw_p99, nv, nskip = train_one_epoch(model, loader, optim)

    train_tag = ''
    if capped < best_train_loss:
        best_train_loss = capped
        torch.save(model.state_dict(), BEST_TRAIN_PT)
        train_tag = ' [↓train]'

    hr_win, hr_ratio = float('nan'), float('nan')
    holdout_tag = ''
    if ep % HOLDOUT_EVERY == 0 or ep == TARGET_EPOCHS:
        hr_win, hr_ratio = evaluate_holdout_gpu(model, holdout)
        if not np.isnan(hr_ratio) and hr_ratio < best_holdout_ratio:
            best_holdout_ratio = hr_ratio
            torch.save(model.state_dict(), BEST_HOLDOUT_PT)
            holdout_tag = ' [↓holdout]'

    trajectory.append({'epoch': ep, 'train_loss': capped,
                       'raw_median': raw_med, 'raw_p99': raw_p99,
                       'holdout_win': hr_win, 'holdout_ratio': hr_ratio})

    if np.isnan(hr_ratio):
        print(f"Epoch {ep:>4d}/{TARGET_EPOCHS} | "
              f"train κ {capped:>7.2f} med {raw_med:>6.1f}{train_tag}")
    else:
        print(f"Epoch {ep:>4d}/{TARGET_EPOCHS} | "
              f"train κ {capped:>7.2f} med {raw_med:>6.1f} | "
              f"holdout win {hr_win:.3f} ratio {hr_ratio:.4f}"
              f"{train_tag}{holdout_tag}")

    # ── Persist after every epoch ────────────────────────────────────
    torch.save({
        'model':              model.state_dict(),
        'opt':                optim.state_dict(),
        'completed':          ep,
        'trajectory':         trajectory,
        'best_holdout_ratio': best_holdout_ratio,
        'best_train_loss':    best_train_loss,
    }, STATE_PT)
    torch.save(model.state_dict(), LATEST_PT)
    with open(TRAJ_PT, 'w') as f:
        json.dump(trajectory, f, indent=2, default=float)

    # ── Periodic snapshot (so mid-run you have backups) ─────────────
    if ep % SNAPSHOT_EVERY == 0:
        torch.save(model.state_dict(),
                   f'{CKPT_DIR}/snapshots/latest_epoch{ep:04d}.pt')
        if best_holdout_ratio < float('inf'):
            torch.save(torch.load(BEST_HOLDOUT_PT, map_location=DEVICE),
                       f'{CKPT_DIR}/snapshots/best_holdout_through_epoch{ep:04d}.pt')

elapsed = time.time() - t0
print('─' * 78)
print(f"Done in {elapsed/60:.1f} min "
      f"({elapsed/(TARGET_EPOCHS - completed):.1f} s/epoch)")
print(f"Best held-out ratio : {best_holdout_ratio:.4f}  → {BEST_HOLDOUT_PT}")
print(f"Best train loss     : {best_train_loss:.4f}  → {BEST_TRAIN_PT}")

## 9 · Plot trajectory

This cell only reads `phase2_trajectory.json` from Drive. You can run it
mid-training, after a disconnect, or after the run finishes — it doesn't
depend on the runtime having the model in memory.

In [ ]:
import matplotlib.pyplot as plt

with open(TRAJ_PT) as f:
    traj = json.load(f)

epochs = [t['epoch'] for t in traj]
train_loss = [t['train_loss'] for t in traj]
ho_epochs    = [t['epoch']         for t in traj if not np.isnan(t['holdout_ratio'])]
ho_ratios    = [t['holdout_ratio'] for t in traj if not np.isnan(t['holdout_ratio'])]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

ax1.plot(epochs, train_loss, color='C0', label='Capped-mean training κ')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Training loss')
ax1.set_title('Training loss across epochs')
ax1.grid(alpha=0.3); ax1.legend()

if ho_ratios:
    ax2.plot(ho_epochs, ho_ratios, 'o-', color='C3', markersize=5,
             label='Median κ ratio (model / chordal)')
    ax2.axhline(1.0, color='gray', linestyle='--', alpha=0.6,
                label='Chordal baseline')
    best_idx = int(np.argmin(ho_ratios))
    ax2.scatter([ho_epochs[best_idx]], [ho_ratios[best_idx]],
                color='red', s=80, zorder=5,
                label=f'Best @ ep {ho_epochs[best_idx]}: ratio {ho_ratios[best_idx]:.3f}')
    ax2.set_title('Held-out N=6 — generalisation curve')
else:
    ax2.text(0.5, 0.5, 'No held-out evaluations yet\n(running, or eval failed)',
             ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('Held-out N=6 (no data)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Median curvature ratio')
ax2.grid(alpha=0.3); ax2.legend()

plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/trajectory.png', dpi=120, bbox_inches='tight')
plt.show()
if ho_ratios:
    print(f"\nBest held-out ratio so far: {min(ho_ratios):.4f} "
          f"at epoch {ho_epochs[int(np.argmin(ho_ratios))]}")

## 10 · Full 500-sample evaluation

Loads the best-by-held-out checkpoint (or best-by-train-loss if held-out
never produced numbers) and runs the proper 500-sample evaluation on
N=4, N=5, N=6.

In [ ]:
# Pick the strongest checkpoint available
import os
if os.path.exists(BEST_HOLDOUT_PT):
    ckpt = BEST_HOLDOUT_PT
    print(f"Loading BEST-BY-HELD-OUT checkpoint: {ckpt}")
elif os.path.exists(BEST_TRAIN_PT):
    ckpt = BEST_TRAIN_PT
    print(f"⚠️  No held-out checkpoint; using BEST-BY-TRAIN-LOSS: {ckpt}")
else:
    ckpt = LATEST_PT
    print(f"⚠️  Using LATEST checkpoint: {ckpt}")

model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
print()

# Build 500-sample test sets
rng = np.random.default_rng(SEED)
_ = gen_random_points(2000, 4, rng)
_ = gen_random_points(2000, 5, rng)
test_2 = gen_random_points(500, 4, rng)
test_3 = gen_random_points(500, 5, rng)
test_4 = gen_random_points(500, 6, rng)

# Full eval — also GPU-aware
def full_eval_gpu(model, points_list, n_dof):
    model.eval()
    kappa_model, kappa_chordal, knot_l1 = [], [], []
    better = valid = 0
    for points_np in points_list:
        chord_knots = ge.chordal_interior_knots(points_np)
        pts_t = torch.tensor(points_np, dtype=torch.double).to(DEVICE)  # GPU fix
        with torch.no_grad():
            pred = model.predict(pts_t, num_knots=n_dof)
        pred_np = pred.cpu().numpy()
        pred_eval = ge.rescale_knots_to_interior(pred_np)
        k_m = ge.compute_curvature_batch(points_np, pred_eval)
        k_c = ge.compute_curvature_batch(points_np, chord_knots)
        if np.isnan(k_m) or np.isnan(k_c): continue
        kappa_model.append(k_m); kappa_chordal.append(k_c)
        knot_l1.append(float(np.mean(np.abs(pred_eval - chord_knots))))
        if k_m <= k_c: better += 1
        valid += 1
    return {
        'n_dof': n_dof, 'n_valid': valid,
        'acc': better / valid if valid else float('nan'),
        'median_model':   float(np.median(kappa_model))   if kappa_model   else float('nan'),
        'median_chordal': float(np.median(kappa_chordal)) if kappa_chordal else float('nan'),
        'mean_knot_l1':   float(np.mean(knot_l1))         if knot_l1       else float('nan'),
    }

results = {}
print(f"{'split':>10} {'N':>3} {'dof':>4} {'win rate':>14} "
      f"{'med model':>11} {'med chord':>11} {'ratio':>7}")
print('─' * 70)
for ndof, tset, split in [(2, test_2, 'TRAIN-N'), (3, test_3, 'TRAIN-N'),
                          (4, test_4, 'HELD-OUT')]:
    r = full_eval_gpu(model, tset, n_dof=ndof)
    nb = int(round(r['acc'] * r['n_valid']))
    ratio = r['median_model'] / r['median_chordal'] if r['median_chordal'] else float('nan')
    print(f"{split:>10} {ndof+2:>3} {ndof:>4}   "
          f"{r['acc']:.3f} ({nb:>3}/{r['n_valid']:>3})  "
          f"{r['median_model']:>10.2f}  {r['median_chordal']:>10.2f}  "
          f"{ratio:>7.4f}")
    results[f'dof{ndof}'] = r

with open(f'{CKPT_DIR}/final_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)
print(f"\nWrote {CKPT_DIR}/final_results.json")

## 11 · If you lose your runtime mid-training

Don't worry — Drive has everything.
1. Reconnect a fresh runtime (GPU)
2. Run cells 1 → 6 (set up env, upload bundle, mount Drive, load data + model)
3. Run **cell 8** — it auto-detects the saved state in Drive and resumes
4. Plot anytime via **cell 9** (reads from Drive, no model state needed)
5. Final eval via **cell 10** (loads checkpoint from Drive)